# Smart Contract Automation Notebook
This notebook automates contract extraction and PDF form generation.

In [134]:
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.generic import NameObject, BooleanObject

In [135]:
# Making sure we are working within the same directory as the jupyter notebook (this will allow for poratbility of the project)
import os
## Verify we're in the right directory
print("Current directory:", os.getcwd())

Current directory: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting


In [136]:
import pandas as pd

# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows (optional — be careful if your BannerPrevious is huge)
pd.set_option('display.max_rows', None)

# Don’t truncate long strings
pd.set_option('display.max_colwidth', None)

# Make the display wider in the notebook
pd.set_option('display.width', 0)

In [137]:
from pathlib import Path


def get_real_excel():
    folder = Path("Step3_input_contract-production_ShariSheets_filled")
    for f in folder.glob("*.xlsx"):
        if f.name.startswith("._"):  # skip macOS junk files
            continue
        return f
    raise FileNotFoundError("No valid Excel file found in Step3_input_contract-production_ShariSheets_filled")


EXCEL_PATH = get_real_excel()




def get_real_pdf(pattern):
    folder = Path("0_source_files/default_PDFcontract")
    for pdf in folder.glob(pattern):
        if pdf.name.startswith("._"):
            continue
        return pdf
    raise FileNotFoundError(f"No valid PDF found for pattern: {pattern}")


print("🔧 USING FILTERED GET_REAL_PDF() — dotfiles will be skipped")


PDF_standard_TEMPLATE_PATH  = get_real_pdf("*standard*.pdf")
PDF_intensive_TEMPLATE_PATH = get_real_pdf("*intensive*.pdf")




#PDF_standard_TEMPLATE_PATH  = next(Path("Step3_input_CONTRACTS_empty").glob("*standard*.pdf"))
#PDF_intensive_TEMPLATE_PATH = next(Path("Step3_input_CONTRACTS_empty").glob("*intensive*.pdf"))


OUTPUT_DIR = Path("Step3_output_contract-production_PDF-CONTRACTS_filled")
OUTPUT_DIR.mkdir(exist_ok=True)


MAX_COURSES_PER_CONTRACT = 3
DEFAULT_RATE_PER_CREDIT = 1050


print("Excel:", EXCEL_PATH)
print("Standard:", PDF_standard_TEMPLATE_PATH)
print("Intensive:", PDF_intensive_TEMPLATE_PATH)




🔧 USING FILTERED GET_REAL_PDF() — dotfiles will be skipped
Excel: Step3_input_contract-production_ShariSheets_filled/ShariSheet_MSSN_FY2027_v20260330.xlsx
Standard: 0_source_files/default_PDFcontract/0_sample_contract_SEM_standard_updated_v20260305_pt8_blue.pdf
Intensive: 0_source_files/default_PDFcontract/0_sample_contract_SEM_intensive_updated_v20260305_pt8_blue.pdf


# What are the fillable fields in the PDF?

In [138]:
from pypdf import PdfReader

reader = PdfReader(PDF_intensive_TEMPLATE_PATH)
fields = reader.get_fields()

print("=== FORM FIELD LIST ===")
for field_name, field_data in fields.items():
    print(f"Field Name : {field_name}")
    print(f"Field Type : {field_data.get('/FT')}")
    print(f"Default    : {field_data.get('/V')}")
    print("-" * 40)


=== FORM FIELD LIST ===
Field Name : last_name
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : first_name
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : id
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : email
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : telephone
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : dept_contact_name
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : dept_contact_id
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : reason
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : course1_code
Field Type : /Tx
Default    : None
----------------------------------------
Field Name : course1_title
Field Type : /Tx
Default    : None
-------------------

In [139]:
from PyPDF2 import PdfReader

reader = PdfReader(PDF_intensive_TEMPLATE_PATH)

fields = reader.get_fields()
print(fields)


{'last_name': {'/T': 'last_name', '/FT': '/Tx', '/TU': 'Last Name'}, 'first_name': {'/T': 'first_name', '/FT': '/Tx', '/TU': 'First Name'}, 'id': {'/T': 'id', '/FT': '/Tx', '/TU': 'ID'}, 'email': {'/T': 'email', '/FT': '/Tx', '/TU': 'Email'}, 'telephone': {'/T': 'telephone', '/FT': '/Tx', '/TU': 'Telephone'}, 'dept_contact_name': {'/T': 'dept_contact_name', '/FT': '/Tx', '/TU': 'Provide dept contact name'}, 'dept_contact_id': {'/T': 'dept_contact_id', '/FT': '/Tx', '/TU': 'ID'}, 'reason': {'/T': 'reason', '/FT': '/Tx', '/TU': 'Reason for Contract (please be specific'}, 'course1_code': {'/T': 'course1_code', '/FT': '/Tx', '/TU': 'Course No._Row_1'}, 'course1_title': {'/T': 'course1_title', '/FT': '/Tx', '/TU': 'Course Title_Row_1'}, 'course1_credits': {'/T': 'course1_credits', '/FT': '/Tx', '/TU': '# of Credits_Row_1'}, 'course2_code': {'/T': 'course2_code', '/FT': '/Tx', '/TU': 'Course No._Row_2'}, 'course2_title': {'/T': 'course2_title', '/FT': '/Tx', '/TU': 'Course Title_Row_2'}, 'co

In [140]:
from PyPDF2 import PdfReader

def get_pdf_fields(pdf_path):
    """
    Print all PDF form fields and their appearance states.
    """
    reader = PdfReader(str(pdf_path))

    if "/AcroForm" not in reader.trailer["/Root"]:
        print("No AcroForm found.")
        return

    form = reader.trailer["/Root"]["/AcroForm"]
    fields = form.get("/Fields", [])

    print("=== PDF FORM FIELDS ===\n")

    for f in fields:
        field = f.get_object()
        name = field.get("/T")
        ftype = field.get("/FT")
        value = field.get("/V")
        kids = field.get("/Kids")

        print(f"FIELD: {name}")
        print(f"  TYPE: {ftype}")
        print(f"  VALUE: {value}")
        print(f"  KIDS: {kids}\n")

        # Show appearance states if present
        ap = field.get("/AP")
        if ap and "/N" in ap:
            print("  --- Appearance States ---")
            for state_name, appearance in ap["/N"].items():
                print(f"    • {state_name}: {appearance}")
        else:
            print("  --- No Appearance States Found ---")

        print("\n" + "-"*50 + "\n")


In [141]:
from PyPDF2 import PdfReader

reader = PdfReader(PDF_standard_TEMPLATE_PATH)

fields = reader.get_fields()
for name, field in fields.items():
    print("FIELD:", name)
    print("  TYPE:", field.get('/FT'))
    print("  VALUE:", field.get('/V'))
    print("  OPTIONS:", field.get('/Opt'))
    print("  STATES:", field.get('/AP'))
    print()


FIELD: last_name
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: first_name
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: id
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: email
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: telephone
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: activity
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: program
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: acct
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: orgn
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: fund
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: amount_total
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: hourly_rate
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: rate_per_credit
  TYPE: /Tx
  VALUE: None
  OPTIONS: None
  STATES: None

FIELD: week_srv_hrs
  TYPE: /Tx
  VALUE: N

# Preparing the DF

In [142]:
def split_name(name):
    if not isinstance(name, str) or "," not in name:
        return name, ""
    last, first = name.split(",", 1)
    return last.strip(), first.strip()

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [143]:
df_raw = pd.read_excel(EXCEL_PATH, sheet_name=0) 
df_raw.head()

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Faculty Name,ID#,email,telephone,status,account to be charged,Reason for Contract,remote employee,international,Dept contact name,Dept contact ID#,Summer 2026 Course,Course Section,Catalog Title,CRN,Term 1,Term 2,Term 3,Mode,Program,Location,load/contract,dept budget,Cr.,rate per credit,total contract amount,Begin Date,End Date,pre-work period start,pre-work period end,pre-work # of weeks,pre-work hours/week,pre-work hours/period,intensive period start,intensive period end,intensive # of weeks,intensive hours/week,intensive hours/period,post-work period start,post-work period end,post-work # of weeks,post-work hours/week,post-work hours/period,sum of weeks (for check),total contract hours,Fall 2026 Course,Course Section.1,Catalog Title.1,CRN.1,Mode.1,Program.1,Location.1,load/contract.1,dept budget.1,Cr..1,rate per credit.1,total contract amount.1,Begin Date.1,End Date.1,pre-work period start.1,pre-work period end.1,pre-work # of weeks.1,pre-work hours/week.1,pre-work hours/period.1,intensive period start.1,intensive period end.1,intensive # of weeks.1,intensive hours/week.1,intensive hours/period.1,post-work period start.1,post-work period end.1,post-work # of weeks.1,post-work hours/week.1,post-work hours/period.1,sum of weeks (for check).1,total contract hours.1,Spring 2027 Course,Course Section.2,Catalog Title.2,CRN.2,Mode.2,Program.2,Location.2,load/contract.2,dept budget.2,Cr..2,rate per credit.2,total contract amount.2,Begin Date.2,End Date.2,pre-work period start.2,pre-work period end.2,pre-work # of weeks.2,pre-work hours/week.2,pre-work hours/period.2,intensive period start.2,intensive period end.2,intensive # of weeks.2,intensive hours/week.2,intensive hours/period.2,post-work period start.2,post-work period end.2,post-work # of weeks.2,post-work hours/week.2,post-work hours/period.2,sum of weeks (for check).2,total contract hours.2,Total Cr.,order
0,Totals: Boubakar Sanou\nnotes: 25% reduced - Department Chair 4,Total Credits: 12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,0.0
1,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN546,NaN,Mission in Cultrl&Rlgs Contxt,NaN,NaN,NaN,NaN,In-Person,MDIV,Main Campus - MC,load,yes,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN632,NaN,NaN,NaN,In-Person,MDIV,Main Campus - MC,load,yes,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN840,NaN,Research Methods in Miss&Min,NaN,NaN,NaN,NaN,In-Person,DIS,Main Campus - MC,load,yes,3,NaN,NaN,2027-06-06 00:00:00,2027-06-14 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN834,NaN,Mission Strategy & Leadership,NaN,NaN,NaN,NaN,In-Person,DIS,Main Campus - MC,load,yes,3,NaN,NaN,2027-05-04 00:00:00,2027-05-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [144]:
list(df_raw.columns)

['Faculty Name',
 'ID#',
 'email',
 'telephone',
 'status',
 'account to be charged',
 'Reason for Contract',
 'remote employee',
 'international',
 'Dept contact name',
 'Dept contact ID#',
 'Summer 2026 Course',
 'Course Section',
 'Catalog Title',
 'CRN',
 'Term 1',
 'Term 2',
 'Term 3',
 'Mode',
 'Program',
 'Location',
 'load/contract',
 'dept budget',
 'Cr.',
 'rate per credit',
 'total contract amount',
 'Begin Date',
 'End Date',
 'pre-work period start',
 'pre-work period end',
 'pre-work # of weeks',
 'pre-work hours/week',
 'pre-work hours/period',
 'intensive period start',
 'intensive period end',
 'intensive # of weeks',
 'intensive hours/week',
 'intensive hours/period',
 'post-work period start',
 'post-work period end',
 'post-work # of weeks',
 'post-work hours/week',
 'post-work hours/period',
 'sum of weeks (for check)',
 'total contract hours',
 'Fall 2026 Course',
 'Course Section.1',
 'Catalog Title.1',
 'CRN.1',
 'Mode.1',
 'Program.1',
 'Location.1',
 'load/con

In [145]:
print(df_raw.columns.tolist())

['Faculty Name', 'ID#', 'email', 'telephone', 'status', 'account to be charged', 'Reason for Contract', 'remote employee', 'international', 'Dept contact name', 'Dept contact ID#', 'Summer 2026 Course', 'Course Section', 'Catalog Title', 'CRN', 'Term 1', 'Term 2', 'Term 3', 'Mode', 'Program', 'Location', 'load/contract', 'dept budget', 'Cr.', 'rate per credit', 'total contract amount', 'Begin Date', 'End Date', 'pre-work period start', 'pre-work period end', 'pre-work # of weeks', 'pre-work hours/week', 'pre-work hours/period', 'intensive period start', 'intensive period end', 'intensive # of weeks', 'intensive hours/week', 'intensive hours/period', 'post-work period start', 'post-work period end', 'post-work # of weeks', 'post-work hours/week', 'post-work hours/period', 'sum of weeks (for check)', 'total contract hours', 'Fall 2026 Course', 'Course Section.1', 'Catalog Title.1', 'CRN.1', 'Mode.1', 'Program.1', 'Location.1', 'load/contract.1', 'dept budget.1', 'Cr..1', 'rate per credit

In [146]:
df_raw.head()

,Faculty Name,ID#,email,telephone,status,account to be charged,Reason for Contract,remote employee,international,Dept contact name,Dept contact ID#,Summer 2026 Course,Course Section,Catalog Title,CRN,Term 1,Term 2,Term 3,Mode,Program,Location,load/contract,dept budget,Cr.,rate per credit,total contract amount,Begin Date,End Date,pre-work period start,pre-work period end,pre-work # of weeks,pre-work hours/week,pre-work hours/period,intensive period start,intensive period end,intensive # of weeks,intensive hours/week,intensive hours/period,post-work period start,post-work period end,post-work # of weeks,post-work hours/week,post-work hours/period,sum of weeks (for check),total contract hours,Fall 2026 Course,Course Section.1,Catalog Title.1,CRN.1,Mode.1,Program.1,Location.1,load/contract.1,dept budget.1,Cr..1,rate per credit.1,total contract amount.1,Begin Date.1,End Date.1,pre-work period start.1,pre-work period end.1,pre-work # of weeks.1,pre-work hours/week.1,pre-work hours/period.1,intensive period start.1,intensive period end.1,intensive # of weeks.1,intensive hours/week.1,intensive hours/period.1,post-work period start.1,post-work period end.1,post-work # of weeks.1,post-work hours/week.1,post-work hours/period.1,sum of weeks (for check).1,total contract hours.1,Spring 2027 Course,Course Section.2,Catalog Title.2,CRN.2,Mode.2,Program.2,Location.2,load/contract.2,dept budget.2,Cr..2,rate per credit.2,total contract amount.2,Begin Date.2,End Date.2,pre-work period start.2,pre-work period end.2,pre-work # of weeks.2,pre-work hours/week.2,pre-work hours/period.2,intensive period start.2,intensive period end.2,intensive # of weeks.2,intensive hours/week.2,intensive hours/period.2,post-work period start.2,post-work period end.2,post-work # of weeks.2,post-work hours/week.2,post-work hours/period.2,sum of weeks (for check).2,total contract hours.2,Total Cr.,order
0,Totals: Boubakar Sanou\nnotes: 25% reduced - Department Chair 4,Total Credits: 12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,0.0
1,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN546,NaN,Mission in Cultrl&Rlgs Contxt,NaN,NaN,NaN,NaN,In-Person,MDIV,Main Campus - MC,load,yes,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN632,NaN,NaN,NaN,In-Person,MDIV,Main Campus - MC,load,yes,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN840,NaN,Research Methods in Miss&Min,NaN,NaN,NaN,NaN,In-Person,DIS,Main Campus - MC,load,yes,3,NaN,NaN,2027-06-06 00:00:00,2027-06-14 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MSSN834,NaN,Mission Strategy & Leadership,NaN,NaN,NaN,NaN,In-Person,DIS,Main Campus - MC,load,yes,3,NaN,NaN,2027-05-04 00:00:00,2027-05-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [147]:
[col for col in df_raw.columns if "Summer" in col or "Fall" in col or "Spring" in col]


['Summer 2026 Course', 'Fall 2026 Course', 'Spring 2027 Course']

# Preparing the Spreadsheet-DF

In [148]:
import pandas as pd
import re

df_raw = pd.read_excel(EXCEL_PATH, sheet_name=0)

# ---------------------------------------------------------
# Utility: detect columns like "Summer 2026"
# ---------------------------------------------------------
def find_real_year_column(df_raw, semester_name):
    """
    Finds a column like 'Summer 2026', 'Summer 2026 Course', etc.
    Returns: (column_name, '2026') or (None, None).
    """
    pattern = re.compile(rf"^{semester_name}\s+(\d{{4}})\b")
    for col in df_raw.columns:
        m = pattern.match(str(col).strip())
        if m:
            return col, m.group(1)
    return None, None


# ---------------------------------------------------------
# Detect terms automatically
# ---------------------------------------------------------
terms = []
for sem in ["Summer", "Fall", "Spring"]:
    col, year = find_real_year_column(df_raw, sem)
    if col:
        terms.append((sem, col, year))

print("Detected term blocks:")
for t in terms:
    print(t)


# ---------------------------------------------------------
# Build suffix map (index-based)
# ---------------------------------------------------------
sem_to_suffix = {
    sem: ("" if idx == 0 else f".{idx}")
    for idx, (sem, col, yr) in enumerate(terms)
}

print("\nSuffix map:")
for sem, col, yr in terms:
    print(f"  {sem}: column={col}, year={yr}, suffix={sem_to_suffix[sem]}")


# ---------------------------------------------------------
# Parse course code like "OTST551 Hebrew I"
# ---------------------------------------------------------
def parse_course(raw):
    if not isinstance(raw, str) or not raw.strip():
        return "", "", ""

    parts = raw.strip().split(maxsplit=1)
    deptnum = parts[0]
    fallback_title = parts[1] if len(parts) > 1 else ""

    m = re.match(r"([A-Za-z]+)(\d+)", deptnum)
    if m:
        dept, num = m.group(1), m.group(2)
    else:
        dept, num = deptnum, ""

    return dept, num, fallback_title


# ---------------------------------------------------------
# Contract row detection
# ---------------------------------------------------------
def is_contract(v):
    """Contract rows must contain ONLY the word 'contract'."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return False
    return str(v).strip().lower() == "contract"


# ---------------------------------------------------------
# NEW: Course Section getter (supports suffix + fallback)
# ---------------------------------------------------------
def get_course_section(row, suffix: str) -> str:
    """
    Prefer a term-specific 'Course Section{suffix}' if present,
    otherwise fall back to plain 'Course Section'.
    Returns a cleaned string (empty if missing).
    """
    # Try suffix version first (only if it exists as a column)
    col_suffix = f"Course Section{suffix}"
    if col_suffix in row.index:
        v = row.get(col_suffix, "")
    else:
        v = row.get("Course Section", "")

    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""

    # Normalize to a clean string; preserve things like "01", "A", "1"
    s = str(v).strip()
    return s


# -----------------------------------------------------
# MAIN EXTRACTION LOOP
# -----------------------------------------------------
records = []

for idx, row in df_raw.iterrows():

    name = row.get("Faculty Name", "")
    if not isinstance(name, str) or "," not in name:
        continue

    instructor = name.strip()

    # Instructor-level NON-term-specific metadata
    base_meta = {
        "Instructor": instructor,
        "ID": row.get("ID#", ""),
        "Email": row.get("email", ""),
        "Telephone": row.get("telephone", ""),
        "Account": row.get("account to be charged", ""),
        "Remote": row.get("remote employee", ""),
        "International": row.get("international", ""),
        "DeptContact": row.get("Dept contact name", ""),
        "DeptContactID": row.get("Dept contact ID#", ""),
        "ChairEmail": row.get("chair email", ""),
        "DeanEmail": row.get("dean email", ""),
        "DeanID": row.get("dean ID#", ""),
        "VPFinanceEmail": row.get("VP finance email", ""),
        "VPFinanceID": row.get("VP finance ID#", ""),
        "HRemail": row.get("HR email", ""),
    }

    # -----------------------------------------------------
    # Process each detected term
    # -----------------------------------------------------
    for (sem, sem_col, year) in terms:

        suffix = sem_to_suffix[sem]
        c_col  = sem_col
        load_c = f"load/contract{suffix}"

        # Contract-only rule
        if not is_contract(row.get(load_c, "")):
            continue

        course_raw = row.get(c_col, "")
        if not isinstance(course_raw, str) or not course_raw.strip():
            continue

        dept, num, fallback_title = parse_course(course_raw)
        course_title = row.get(f"Catalog Title{suffix}", fallback_title)

        # NEW: course section pulled from df_raw
        course_section = get_course_section(row, suffix)

        # -----------------------------------------------------
        # BUILD TERM-SPECIFIC COMBINED REASON
        # -----------------------------------------------------
        prog = row.get(f"Program{suffix}", "")
        loc  = row.get(f"Location{suffix}", "")
        reason_raw = row.get("Reason for Contract", "")

        reason_parts = []
        for item in (prog, loc, reason_raw):
            if isinstance(item, str) and item.strip():
                reason_parts.append(item.strip())

        combined_reason = "/".join(reason_parts)

        # -----------------------------------------------------
        # Create the record for this term
        # -----------------------------------------------------
        records.append({
            **base_meta,
            "Reason": combined_reason,
            "Semester": sem,
            "Year": year,
            "CourseRaw": course_raw,
            "Dept": dept,
            "CourseNum": num,
            "CourseSection": course_section,   # <-- ADDED
            "CourseTitle": course_title,

            "Credits": row.get(f"Cr.{suffix}", None),
            "Rate": row.get(f"rate per credit{suffix}", None),
            "DeptBudget": row.get(f"dept budget{suffix}", ""),

            "BeginDate": row.get(f"Begin Date{suffix}", None),
            "EndDate":   row.get(f"End Date{suffix}", None),

            "prework_start":        row.get(f"pre-work period start{suffix}", None),
            "prework_end":          row.get(f"pre-work period end{suffix}", None),
            "prework_weeks":        row.get(f"pre-work # of weeks{suffix}", None),
            "prework_hours_week":   row.get(f"pre-work hours/week{suffix}", None),
            "prework_hours_period": row.get(f"pre-work hours/period{suffix}", None),

            "intensive_start": row.get(f"intensive period start{suffix}", None),
            "intensive_end":   row.get(f"intensive period end{suffix}", None),
            "intensive_weeks": row.get(f"intensive # of weeks{suffix}", None),
            "intensive_hours_week":   row.get(f"intensive hours/week{suffix}", None),
            "intensive_hours_period": row.get(f"intensive hours/period{suffix}", None),

            "postwork_start": row.get(f"post-work period start{suffix}", None),
            "postwork_end":   row.get(f"post-work period end{suffix}", None),
            "postwork_weeks": row.get(f"post-work # of weeks{suffix}", None),
            "postwork_hours_week":   row.get(f"post-work hours/week{suffix}", None),
            "postwork_hours_period": row.get(f"post-work hours/period{suffix}", None),
        })


# ---------------------------------------------------------
# Final DF
# ---------------------------------------------------------
df_courses = pd.DataFrame(records)
print("\nExtracted contract rows:", len(df_courses))
df_courses.head()

Detected term blocks:
('Summer', 'Summer 2026 Course', '2026')
('Fall', 'Fall 2026 Course', '2026')
('Spring', 'Spring 2027 Course', '2027')

Suffix map:
  Summer: column=Summer 2026 Course, year=2026, suffix=
  Fall: column=Fall 2026 Course, year=2026, suffix=.1
  Spring: column=Spring 2027 Course, year=2027, suffix=.2

Extracted contract rows: 13


/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Instructor,ID,Email,Telephone,Account,Remote,International,DeptContact,DeptContactID,ChairEmail,DeanEmail,DeanID,VPFinanceEmail,VPFinanceID,HRemail,Reason,Semester,Year,CourseRaw,Dept,CourseNum,CourseSection,CourseTitle,Credits,Rate,DeptBudget,BeginDate,EndDate,prework_start,prework_end,prework_weeks,prework_hours_week,prework_hours_period,intensive_start,intensive_end,intensive_weeks,intensive_hours_week,intensive_hours_period,postwork_start,postwork_end,postwork_weeks,postwork_hours_week,postwork_hours_period
0,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Pacific Union - PU,Fall,2026,MSSN505,MSSN,505,,Christn Respns to Human Needs,3.0,1050,yes,2027-09-15,2027-09-19,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
1,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Southwestern Union - SWU,Spring,2027,MSSN546,MSSN,546,,Mission in Cultrl&Rlgs Contxt,3.0,1050,yes,2027-01-10,2027-01-14,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
2,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Lake Union- LU,Fall,2026,MSSN546,MSSN,546,,Mission in Cultrl&Rlgs Contxt,3.0,1050,yes,2027-10-04,2027-10-08,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
3,"Pavel, Zubkov",NaN,NaN,NaN,MSSN: 11-1200-9120-70-120011,NaN,NaN,NaN,NaN,,,,,,,DIS/Main Campus - MC,Summer,2026,MSSN838,MSSN,838,001,Mission & the Social Sciences,3.0,1050,yes,2026-05-11,2027-02-27,2026-05-11,2026-06-03,4.0,1.0,4.0,2026-06-04,2026-06-12,2.0,20,40.0,2026-06-13,2027-02-24,37.0,3.0,64.0
4,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,MSSN: 11-1200-9120-70-120011,NaN,NaN,NaN,NaN,,,,,,,DIS/Main Campus - MC,Summer,2026,MSSN885,MSSN,885,001,Intercultural Mission Leadership,3.0,1050,yes,2026-05-11,2027-02-24,2026-05-11,2026-06-14,5.0,1.0,5.0,2026-06-15,2026-06-23,2.0,30,60.0,2026-06-24,2027-02-24,35.0,3.0,43.0


In [149]:
print(df_raw.filter(like="load/contract").head(20))


   load/contract load/contract.1 load/contract.2
0            NaN             NaN             NaN
1           load             NaN            load
2           load             NaN             NaN
3           load             NaN             NaN
4            NaN             NaN             NaN
5            NaN             NaN             NaN
6            NaN             NaN             NaN
7            NaN             NaN             NaN
8            NaN             NaN             NaN
9            NaN             NaN             NaN
10           NaN             NaN             NaN
11          load             NaN             NaN
12          load             NaN             NaN
13          load             NaN             NaN
14          load             NaN             NaN
15           NaN             NaN             NaN
16           NaN             NaN             NaN
17           NaN             NaN             NaN
18           NaN             NaN             NaN
19           NaN    

In [150]:
df_courses.columns.tolist()

['Instructor',
 'ID',
 'Email',
 'Telephone',
 'Account',
 'Remote',
 'International',
 'DeptContact',
 'DeptContactID',
 'ChairEmail',
 'DeanEmail',
 'DeanID',
 'VPFinanceEmail',
 'VPFinanceID',
 'HRemail',
 'Reason',
 'Semester',
 'Year',
 'CourseRaw',
 'Dept',
 'CourseNum',
 'CourseSection',
 'CourseTitle',
 'Credits',
 'Rate',
 'DeptBudget',
 'BeginDate',
 'EndDate',
 'prework_start',
 'prework_end',
 'prework_weeks',
 'prework_hours_week',
 'prework_hours_period',
 'intensive_start',
 'intensive_end',
 'intensive_weeks',
 'intensive_hours_week',
 'intensive_hours_period',
 'postwork_start',
 'postwork_end',
 'postwork_weeks',
 'postwork_hours_week',
 'postwork_hours_period']

In [151]:
df_courses.head()

,Instructor,ID,Email,Telephone,Account,Remote,International,DeptContact,DeptContactID,ChairEmail,DeanEmail,DeanID,VPFinanceEmail,VPFinanceID,HRemail,Reason,Semester,Year,CourseRaw,Dept,CourseNum,CourseSection,CourseTitle,Credits,Rate,DeptBudget,BeginDate,EndDate,prework_start,prework_end,prework_weeks,prework_hours_week,prework_hours_period,intensive_start,intensive_end,intensive_weeks,intensive_hours_week,intensive_hours_period,postwork_start,postwork_end,postwork_weeks,postwork_hours_week,postwork_hours_period
0,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Pacific Union - PU,Fall,2026,MSSN505,MSSN,505,,Christn Respns to Human Needs,3.0,1050,yes,2027-09-15,2027-09-19,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
1,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Southwestern Union - SWU,Spring,2027,MSSN546,MSSN,546,,Mission in Cultrl&Rlgs Contxt,3.0,1050,yes,2027-01-10,2027-01-14,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
2,"Duffis Gordon, Daniel M.",NaN,duffisgordo@andrews.edu,NaN,MSSN: 11-1200-9120-70,NaN,NaN,"Sanou, Boubakar",000098144,,,,,,,MAPM-HISP/Lake Union- LU,Fall,2026,MSSN546,MSSN,546,,Mission in Cultrl&Rlgs Contxt,3.0,1050,yes,2027-10-04,2027-10-08,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
3,"Pavel, Zubkov",NaN,NaN,NaN,MSSN: 11-1200-9120-70-120011,NaN,NaN,NaN,NaN,,,,,,,DIS/Main Campus - MC,Summer,2026,MSSN838,MSSN,838,001,Mission & the Social Sciences,3.0,1050,yes,2026-05-11,2027-02-27,2026-05-11,2026-06-03,4.0,1.0,4.0,2026-06-04,2026-06-12,2.0,20,40.0,2026-06-13,2027-02-24,37.0,3.0,64.0
4,"Sanou, Boubakar",000098144,sanou@andrews.edu,NaN,MSSN: 11-1200-9120-70-120011,NaN,NaN,NaN,NaN,,,,,,,DIS/Main Campus - MC,Summer,2026,MSSN885,MSSN,885,001,Intercultural Mission Leadership,3.0,1050,yes,2026-05-11,2027-02-24,2026-05-11,2026-06-14,5.0,1.0,5.0,2026-06-15,2026-06-23,2.0,30,60.0,2026-06-24,2027-02-24,35.0,3.0,43.0


# Matching DF fields with PDF fields

In [152]:
# --- Cell 6: Build PDF field data + PDF filler function ---  (UPDATED: ONE COURSE PER PDF + GENERATE ALL CONTRACTS)

import math
import re
from pathlib import Path
import pandas as pd

# NOTE: Use NameObject from pypdf (you already use pypdf below)
from pypdf.generic import NameObject, TextStringObject, BooleanObject


# --------------------------------------------------------
# Utilities
# --------------------------------------------------------

def safe_int_string(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    try:
        return str(int(float(str(x).strip())))
    except Exception:
        return ""

def safe_number(x):
    if x is None:
        return 0
    try:
        if pd.isna(x):
            return 0
    except Exception:
        pass
    try:
        return float(str(x).strip())
    except Exception:
        return 0

def normalize_date(raw):
    if raw is None or raw == "" or (isinstance(raw, float) and math.isnan(raw)):
        return ""
    if hasattr(raw, "strftime") and not pd.isna(raw):
        return raw.strftime("%m/%d/%Y")
    s = str(raw).strip()
    if "-" in s and len(s.split("-")) >= 3:
        try:
            y, m, d = s.split("-")[:3]
            return f"{m.zfill(2)}/{d.zfill(2)}/{y}"
        except Exception:
            pass
    if "/" in s:
        return s
    return ""

def split_mmddyyyy(raw):
    if not raw:
        return "", "", ""
    parts = str(raw).strip().split("/")
    if len(parts) != 3:
        return "", "", ""
    mm, dd, yyyy = parts
    return mm.zfill(2), dd.zfill(2), yyyy[-2:]

def format_phone(raw):
    if raw is None:
        return ""
    digits = re.sub(r"\D", "", str(raw))
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    if len(digits) == 11 and digits.startswith("1"):
        d = digits[1:]
        return f"({d[:3]}) {d[3:6]}-{d[6:]}"
    return ""

def format_year_two_digits(raw):
    if raw is None:
        return ""
    s = str(raw).strip()
    return s[-2:] if s[-2:].isdigit() else ""

def format_id(raw):
    if raw is None:
        return ""
    s = str(raw).strip()
    if s.lower() == "nan":
        return ""
    if s.endswith(".0"):
        s = s[:-2]
    if re.fullmatch(r"\d+\.\d+", s):
        try:
            return str(int(float(s)))
        except Exception:
            return s
    return s


# --------------------------------------------------------
# Course field mapping (UPDATED: ONE COURSE PER PDF)
# --------------------------------------------------------

def map_course_fields(course):
    """
    Convert exactly ONE course into PDF field names.
    Adds CourseSection to course_code when present: e.g., CHIS570-999

    Fills course1_* only, and blanks out course2/course3 fields
    (safe even if those fields don't exist in the PDF).
    """
    c = course or {}
    pdf_fields = {}

    dept = str(c.get("Dept", "") or "").strip()
    num  = str(c.get("CourseNum", "") or "").strip()
    sec  = str(c.get("CourseSection", "") or "").strip()

    if sec.lower() == "nan":
        sec = ""

    course_code = f"{dept}{num}" if (dept or num) else str(c.get("CourseRaw", "") or "").strip()
    if course_code and sec:
        course_code = f"{course_code}-{sec}"

    pdf_fields["course1_code"]       = course_code
    pdf_fields["course1_title"]      = c.get("CourseTitle", "")
    pdf_fields["course1_credits"]    = c.get("Credits")
    pdf_fields["course1_enrollment"] = c.get("Enrollment")

    # Explicitly blank any extra course slots (in case the PDF has them)
    pdf_fields["course2_code"] = ""
    pdf_fields["course2_title"] = ""
    pdf_fields["course2_credits"] = ""
    pdf_fields["course2_enrollment"] = ""
    pdf_fields["course3_code"] = ""
    pdf_fields["course3_title"] = ""
    pdf_fields["course3_credits"] = ""
    pdf_fields["course3_enrollment"] = ""

    return pdf_fields


def parse_account_number(raw):
    if not isinstance(raw, str):
        return "", "", "", "", ""
    if ":" in raw:
        raw = raw.split(":", 1)[1].strip()

    parts = raw.split("-")
    parts += [""] * (4 - len(parts))

    fund    = parts[0].strip()
    orgn    = parts[1].strip()
    acct    = parts[2].strip()
    program = parts[3].strip()
    activity = parts[4].strip() if len(parts) > 4 else ""

    return fund, orgn, acct, program, activity


# --------------------------------------------------------
# Formatting Tweaks (recursive field tree + widget styling)
# --------------------------------------------------------

def _walk_field_tree(obj):
    """Recursively yield all nodes under a field tree."""
    yield obj
    kids = obj.get("/Kids")
    if kids:
        for kid in kids:
            try:
                kid_obj = kid.get_object()
            except Exception:
                continue
            yield from _walk_field_tree(kid_obj)

def _is_widget(obj):
    return obj.get("/Subtype") == NameObject("/Widget") or obj.get("/Subtype") == "/Widget"

def _resolve_ft(node):
    """Resolve /FT from node or its parent (some widgets inherit it)."""
    ft = node.get("/FT")
    if ft is None and node.get("/Parent"):
        try:
            ft = node["/Parent"].get_object().get("/FT")
        except Exception:
            ft = None
    return ft

def apply_text_style_to_text_fields_only(
    acroform,
    font_name="/STSong-Light",  # alternatively /Helvetica-Bold
    font_size=8,
    rgb=(0, 0, 1),
    remove_text_ap=True,
):
    """
    Apply fixed font+size+color ONLY to text fields (/FT /Tx) across the entire tree.
    Never touch /Btn fields (checkbox/radio).
    Optionally remove /AP only for text widgets (safe).
    """
    r, g, b = rgb
    da = f"{r} {g} {b} rg {font_name} {font_size} Tf"

    # Strengthen default too
    acroform[NameObject("/DA")] = TextStringObject(da)

    for f in acroform.get("/Fields", []):
        root = f.get_object()
        for node in _walk_field_tree(root):
            ft = _resolve_ft(node)
            is_text = (ft == NameObject("/Tx") or ft == "/Tx")

            if is_text:
                node[NameObject("/DA")] = TextStringObject(da)

            # Only remove /AP for TEXT widgets, and only if requested
            if remove_text_ap and is_text and _is_widget(node):
                if "/AP" in node:
                    del node["/AP"]


# --------------------------------------------------------
# Build the PDF data dictionary (UPDATED: ONE COURSE PER PDF)
# --------------------------------------------------------

def make_pdf_data_for_contract(instructor_name, semester, year, courses_chunk):
    """
    This ALWAYS builds data for exactly ONE course per PDF.
    If a list is passed, only the FIRST row is used.
    """

    try:
        last, first = split_name(instructor_name)
    except Exception:
        last, first = instructor_name, ""

    # Use only first course
    meta = courses_chunk[0] if courses_chunk else {}

    def safe_credit(x):
        try:
            x = float(x)
            return 0 if math.isnan(x) else x
        except Exception:
            return 0

    total_credits = safe_credit(meta.get("Credits"))
    weekly_hours = total_credits * 3

    raw_rate = meta.get("Rate")
    if raw_rate is None or str(raw_rate).strip() in ("", "nan"):
        rate_per_credit = DEFAULT_RATE_PER_CREDIT
    else:
        try:
            rate_per_credit = float(raw_rate)
        except Exception:
            rate_per_credit = raw_rate

    if isinstance(rate_per_credit, (int, float)) and total_credits:
        amount_value = rate_per_credit * total_credits
    else:
        # fallback ONLY from this course
        if isinstance(meta.get("Salary"), (int, float)):
            amount_value = meta.get("Salary")
        elif isinstance(meta.get("PhD"), (int, float)):
            amount_value = meta.get("PhD")
        else:
            amount_value = ""

    amount_str = (
        str(int(amount_value))
        if isinstance(amount_value, (int, float)) and not math.isnan(amount_value)
        else ""
    )

    begin_raw = normalize_date(meta.get("BeginDate"))
    end_raw   = normalize_date(meta.get("EndDate"))

    begin_m, begin_d, begin_y = split_mmddyyyy(begin_raw)
    end_m,   end_d,   end_y   = split_mmddyyyy(end_raw)

    acct_raw = meta.get("Account", "")
    fund, orgn, acct_num, program, activity = parse_account_number(acct_raw)

    data = {
        "last_name": last,
        "first_name": first,
        "id": format_id(meta.get("ID")),
        "email": meta.get("Email", ""),
        "telephone": format_phone(meta.get("Telephone")),

        "remote_yes": "Off",
        "payment_yes": "Off",
        "payment_no": "Off",

        "dept_contact_name": meta.get("DeptContact", ""),
        "dept_contact_id": format_id(meta.get("DeptContactID")),

        "reason": meta.get("Reason", ""),

        "sem_spring": "Off",
        "sem_summer": "Off",
        "sem_fall":   "Off",

        "year": format_year_two_digits(year),

        "begin_month": begin_m,
        "begin_day": begin_d,
        "begin_year": begin_y,

        "end_month": end_m,
        "end_day": end_d,
        "end_year": end_y,

        "Begin Date month": begin_m,
        "Begin Date date": begin_d,
        "Begin Date year": begin_y,
        "End Date month": end_m,
        "End Date date": end_d,
        "End Date year": end_y,

        "credits_total": str(int(total_credits)) if total_credits else "",
        "week_srv_hrs": str(int(weekly_hours)) if weekly_hours else "",
        "rate_per_credit": str(rate_per_credit),
        "hourly_rate": "",
        "amount_total": amount_str,

        "fund": fund,
        "orgn": orgn,
        "acct": acct_num,
        "program": program,
        "activity": activity,

        "Signature2ID": format_id(meta.get("DeptContactID")),
        "Signature3ID": format_id(meta.get("DeanID")),
        "Signature4ID": format_id(meta.get("VPFinanceID")),
    }

    # Flags (ONLY from this one course/meta row)
    remote_raw = str(meta.get("Remote", "")).strip().lower()
    data["remote_yes"] = "/Yes" if remote_raw in ("yes", "y", "true", "1") else "/Off"

    sem_raw = str(meta.get("Semester", "")).strip().lower()
    data["spring_yes"] = NameObject("/Yes") if sem_raw == "spring" else NameObject("/Off")
    data["summer_yes"] = NameObject("/Yes") if sem_raw == "summer" else NameObject("/Off")
    data["fall_yes"]   = NameObject("/Yes") if sem_raw == "fall"   else NameObject("/Off")

    international_raw = str(meta.get("International", "")).strip().lower()
    data["international"] = "/Yes" if international_raw in ("yes", "y", "true", "1") else "/Off"

    dept_budget_raw = str(meta.get("DeptBudget", "")).strip().lower()
    data["dept_budget_yes"] = "/Yes" if dept_budget_raw in ("yes", "y", "true", "1") else "/Off"
    data["dept_budget_no"]  = "/Yes" if dept_budget_raw in ("no", "n", "false", "0") else "/Off"

    # Course fields (ONLY ONE course; includes CourseSection)
    data.update(map_course_fields(meta))

    # Contract timing fields (from single course/meta)
    data["prework_start"]        = normalize_date(meta.get("prework_start"))
    data["prework_end"]          = normalize_date(meta.get("prework_end"))
    data["prework_of_weeks"]     = safe_int_string(meta.get("prework_weeks"))
    data["prework_hours_weeks"]  = safe_int_string(meta.get("prework_hours_week"))
    data["prework_hours_period"] = safe_int_string(meta.get("prework_hours_period"))

    data["intensive_start"]        = normalize_date(meta.get("intensive_start"))
    data["intensive_end"]          = normalize_date(meta.get("intensive_end"))
    data["intensive_of_weeks"]     = safe_int_string(meta.get("intensive_weeks"))
    data["intensive_hours_weeks"]  = safe_int_string(meta.get("intensive_hours_week"))
    data["intensive_hours_period"] = safe_int_string(meta.get("intensive_hours_period"))

    data["postwork_start"]        = normalize_date(meta.get("postwork_start"))
    data["postwork_end"]          = normalize_date(meta.get("postwork_end"))
    data["postwork_of_weeks"]     = safe_int_string(meta.get("postwork_weeks"))
    data["postwork_hours_week"]   = safe_int_string(meta.get("postwork_hours_week"))
    data["postwork_hours_period"] = safe_int_string(meta.get("postwork_hours_period"))

    pre_h  = safe_number(meta.get("prework_hours_period"))
    int_h  = safe_number(meta.get("intensive_hours_period"))
    post_h = safe_number(meta.get("postwork_hours_period"))
    data["total_contract_hours"] = safe_int_string(pre_h + int_h + post_h)

    if meta:
        print(
            f"[DEBUG] {instructor_name}, {semester} {year} – "
            f"BeginDate={meta.get('BeginDate')!r} -> {begin_m}/{begin_d}/{begin_y}, "
            f"EndDate={meta.get('EndDate')!r} -> {end_m}/{end_d}/{end_y}, "
            f"course1_code={data.get('course1_code')!r}"
        )

    return data


# --------------------------------------------------------
# PDF Writer
# --------------------------------------------------------

def sanitize_pdf_data(data: dict):
    clean = {}
    for k, v in data.items():
        if isinstance(v, NameObject):
            clean[k] = v
            continue
        if v is None:
            clean[k] = ""
            continue
        try:
            if isinstance(v, float) and math.isnan(v):
                clean[k] = ""
            else:
                clean[k] = str(v)
        except Exception:
            clean[k] = ""
    return clean


from pypdf import PdfReader, PdfWriter

def fill_pdf_contract(template_path, output_path, data):
    clean_data = sanitize_pdf_data(data)

    reader = PdfReader(str(template_path))
    writer = PdfWriter()

    for page in reader.pages:
        writer.add_page(page)

    acroform = None
    if "/AcroForm" in reader.trailer["/Root"]:
        acroform = reader.trailer["/Root"]["/AcroForm"]
        writer._root_object.update({NameObject("/AcroForm"): acroform})

        try:
            writer.set_need_appearances_writer(True)
        except Exception:
            acroform.get_object().update({NameObject("/NeedAppearances"): BooleanObject(True)})

    # Apply DA before filling
    if acroform is not None:
        acro_obj = acroform.get_object()
        acro_obj.update({NameObject("/NeedAppearances"): BooleanObject(True)})
        apply_text_style_to_text_fields_only(
            acro_obj,
            font_name="/STSong-Light",
            font_size=8,
            rgb=(0, 0, 1),
            remove_text_ap=False,
        )

    # Fill values
    for page in writer.pages:
        writer.update_page_form_field_values(
            page,
            clean_data,
            auto_regenerate=True,
        )

    # Remove /AP for text widgets only after filling
    if acroform is not None:
        acro_obj = acroform.get_object()
        apply_text_style_to_text_fields_only(
            acro_obj,
            font_name="/STSong-Light",
            font_size=8,
            rgb=(0, 0, 1),
            remove_text_ap=True,
        )
        acro_obj.update({NameObject("/NeedAppearances"): BooleanObject(True)})

    with open(output_path, "wb") as f:
        writer.write(f)


# --------------------------------------------------------
# NEW: Generate ALL contracts from df_course (ONE PDF PER ROW)
# --------------------------------------------------------

def _build_course_code_from_row(r: dict) -> str:
    dept = str(r.get("Dept", "") or "").strip()
    num  = str(r.get("CourseNum", "") or "").strip()
    sec  = str(r.get("CourseSection", "") or "").strip()
    if sec.lower() == "nan":
        sec = ""
    code = f"{dept}{num}" if (dept or num) else str(r.get("CourseRaw", "") or "").strip()
    if code and sec:
        code = f"{code}-{sec}"
    return code or "COURSE"

def _safe_filename(s: str) -> str:
    s = str(s or "").strip()
    s = re.sub(r"[^\w\-\. ]+", "", s)      # drop odd chars
    s = re.sub(r"\s+", "_", s).strip("_")  # spaces -> underscores
    return s or "UNKNOWN"

def generate_all_contract_pdfs_one_per_course(
    df_course: pd.DataFrame,
    template_path,
    output_dir,
    instructor_col="Instructor",   # change if needed
    semester_col="Semester",       # change if needed
    year_value=None,               # e.g., FY, or None if year is in the df
    year_col="Year",               # used only if year_value is None
    filename_add_row_index=True,   # keeps filenames unique even if duplicates exist
):
    """
    CRITICAL FIX:
    This writes ONE PDF per row in df_course (per course),
    so if Professor A teaches 3 summer courses -> 3 PDFs.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    written = 0

    for idx, row in df_course.iterrows():
        r = row.to_dict()

        instructor = r.get(instructor_col, "") or ""
        semester   = r.get(semester_col, "") or ""
        year       = year_value if year_value is not None else (r.get(year_col, "") or "")

        # Build ONE-course payload (pass [r] so the function stays compatible)
        data = make_pdf_data_for_contract(instructor, semester, year, [r])

        # Build unique filename
        try:
            last, first = split_name(instructor)
        except Exception:
            last, first = instructor, ""

        course_code = _build_course_code_from_row(r)

        base_name = (
            f"{_safe_filename(last)}_{_safe_filename(first)}__"
            f"{_safe_filename(str(semester))}_{_safe_filename(str(year))}__"
            f"{_safe_filename(course_code)}"
        )
        if filename_add_row_index:
            base_name += f"__row{idx}"

        out_path = output_dir / f"{base_name}.pdf"

        fill_pdf_contract(template_path, out_path, data)
        written += 1

    print(f"[DONE] Wrote {written} contract PDFs to: {output_dir}")

In [153]:
# === Helper for Cell A/B ===

import re

def _clean_str(x):
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() == "nan":
        return ""
    return s

def _clean_section(sec):
    s = _clean_str(sec)
    # normalize common section formats; keep letters/numbers
    s = s.replace("/", "-")
    s = re.sub(r"\s+", "", s)
    return s

def course_token(course_dict):
    """
    Build token like 'OTST551-01' from a course record dict.
    Expects keys: Dept, CourseNum, CourseSection
    """
    dept = _clean_str(course_dict.get("Dept", ""))
    num  = _clean_str(course_dict.get("CourseNum", ""))
    sec  = _clean_section(course_dict.get("CourseSection", ""))

    base = f"{dept}{num}".strip()
    if not base:
        # fallback: try parsing CourseRaw if needed
        raw = _clean_str(course_dict.get("CourseRaw", ""))
        base = raw.split()[0] if raw else "UNKNOWN"

    return f"{base}-{sec}" if sec else base

def course_tokens_for_chunk(chunk):
    """
    chunk is a list[dict] (from to_dict("records"))
    Returns stable, de-duplicated token string joined by '+'
    """
    toks = [course_token(c) for c in chunk]
    # de-dupe preserving order
    seen = set()
    toks2 = []
    for t in toks:
        if t and t not in seen:
            seen.add(t)
            toks2.append(t)
    return "+".join(toks2) if toks2 else "NOCOURSE"


In [154]:
# === Cell A: Generate STANDARD contracts only ===
# UPDATED: ONE COURSE PER PDF (no chunking); generates ALL contracts

semester_code = {"Spring": "01", "Summer": "02", "Fall": "03"}
generated_standard_files = []

grouped = df_courses.groupby(["Instructor", "Semester", "Year"])

for (instructor, semester, year), subdf in grouped:

    # ---- Split intensive vs standard (STANDARD = no intensive_start) ----
    standard_df = subdf[
        subdf["intensive_start"].isna() |
        (subdf["intensive_start"].astype(str).str.strip() == "")
    ]

    # ONE PDF PER ROW/COURSE
    if standard_df.empty:
        continue

    last, first = split_name(instructor)
    safe_name = f"{first}_{last}".replace(" ", "_")
    sem_code = semester_code.get(semester, "00")

    # ----------------------------------------------------
    # PROCESS STANDARD CONTRACTS (ONE COURSE PER PDF)
    # ----------------------------------------------------
    for row_i, row in standard_df.reset_index(drop=False).iterrows():
        c = row.to_dict()

        # Optional: per-course debug (uncomment if needed)
        # print("CourseRaw:", c.get("CourseRaw"))
        # print("Dept:", c.get("Dept"), "CourseNum:", c.get("CourseNum"), "CourseSection:", c.get("CourseSection"))
        # print("----")

        # IMPORTANT: pass ONE course only
        data = make_pdf_data_for_contract(instructor, semester, year, [c])

        # Filename token for this single course
        token_str = course_token(c) if "course_token" in globals() else course_tokens_for_chunk([c])

        base = f"{year}-{sem_code}_{semester}_standard_{safe_name}_{token_str}"

        # Ensure uniqueness (same course token could appear multiple times)
        # Prefer original dataframe index if present; fallback to row number
        unique_suffix = c.get("index", None)
        if unique_suffix is None or str(unique_suffix).strip() == "":
            unique_suffix = row_i
        base = f"{base}_row{unique_suffix}"

        outfile = OUTPUT_DIR / f"{base}.pdf"
        fill_pdf_contract(PDF_standard_TEMPLATE_PATH, outfile, data)
        generated_standard_files.append(outfile)

len(generated_standard_files), generated_standard_files[:10]

[DEBUG] Duffis Gordon, Daniel M., Fall 2026 – BeginDate=Timestamp('2027-09-15 00:00:00') -> 09/15/27, EndDate=Timestamp('2027-09-19 00:00:00') -> 09/19/27, course1_code='MSSN505'
[DEBUG] Duffis Gordon, Daniel M., Fall 2026 – BeginDate=Timestamp('2027-10-04 00:00:00') -> 10/04/27, EndDate=Timestamp('2027-10-08 00:00:00') -> 10/08/27, course1_code='MSSN546'
[DEBUG] Duffis Gordon, Daniel M., Spring 2027 – BeginDate=Timestamp('2027-01-10 00:00:00') -> 01/10/27, EndDate=Timestamp('2027-01-14 00:00:00') -> 01/14/27, course1_code='MSSN546'
[DEBUG] Goya, Ignacio, Spring 2027 – BeginDate=NaT -> //, EndDate=NaT -> //, course1_code='MSSN625'
[DEBUG] Hunter, Claval, Fall 2026 – BeginDate=NaT -> //, EndDate=NaT -> //, course1_code='MSSN667'
[DEBUG] Kwon, Sung K., Fall 2026 – BeginDate=NaT -> //, EndDate=NaT -> //, course1_code='MSSN644'
[DEBUG] Kwon, Sung K., Summer 2026 – BeginDate=Timestamp('2026-08-03 00:00:00') -> 08/03/26, EndDate=Timestamp('2026-08-10 00:00:00') -> 08/10/26, course1_code='MSS

(9,
 [PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-03_Fall_standard_Daniel_M._Duffis_Gordon_MSSN505_row0.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-03_Fall_standard_Daniel_M._Duffis_Gordon_MSSN546_row2.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2027-01_Spring_standard_Daniel_M._Duffis_Gordon_MSSN546_row1.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2027-01_Spring_standard_Ignacio_Goya_MSSN625_row12.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-03_Fall_standard_Claval_Hunter_MSSN667_row10.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-03_Fall_standard_Sung_K._Kwon_MSSN644_row8.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-02_Summer_standard_Sung_K._Kwon_MSSN655_row7.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-03_Fall_standard_Luis_F._Ort

In [155]:
# === Cell B: Generate INTENSIVE contracts only ===
# UPDATED: ONE COURSE PER PDF (no chunking); generates ALL contracts

semester_code = {"Spring": "01", "Summer": "02", "Fall": "03"}
generated_intensive_files = []

grouped = df_courses.groupby(["Instructor", "Semester", "Year"])

for (instructor, semester, year_full), subdf in grouped:

    # ---- Split intensive vs standard (INTENSIVE = has intensive_start) ----
    intensive_df = subdf[
        subdf["intensive_start"].notna() &
        (subdf["intensive_start"].astype(str).str.strip() != "")
    ]

    # ONE PDF PER ROW/COURSE
    if intensive_df.empty:
        continue

    last, first = split_name(instructor)
    safe_name = f"{first}_{last}".replace(" ", "_")
    sem_code = semester_code.get(semester, "00")

    # ----------------------------------------------------
    # PROCESS INTENSIVE CONTRACTS (ONE COURSE PER PDF)
    # ----------------------------------------------------
    for row_i, row in intensive_df.reset_index(drop=False).iterrows():
        c = row.to_dict()

        # IMPORTANT: pass ONE course only
        data = make_pdf_data_for_contract(instructor, semester, year_full, [c])

        # Filename token for this single course
        token_str = course_token(c) if "course_token" in globals() else course_tokens_for_chunk([c])

        base = f"{year_full}-{sem_code}_{semester}_intensive_{safe_name}_{token_str}"

        # Ensure uniqueness
        unique_suffix = c.get("index", None)
        if unique_suffix is None or str(unique_suffix).strip() == "":
            unique_suffix = row_i
        base = f"{base}_row{unique_suffix}"

        outfile = OUTPUT_DIR / f"{base}.pdf"

        # Optional detailed debug (now single-course)
        if instructor == "Glanz, Oliver" and semester == "Summer":
            print("\n🔍 DEBUG for Summer Intensive — Glanz, Oliver")
            print("course:", c.get("CourseRaw"), "| CourseToken =", token_str, "| postwork_end =", c.get("postwork_end"))
            print("PDF data postwork_end:", data.get("postwork_end"))
            print("PDF data keys with 'postwork':", [k for k in data.keys() if "postwork" in k.lower()])
            print("Account fields:", {
                "fund": data.get("fund"),
                "orgn": data.get("orgn"),
                "acct": data.get("acct"),
                "program": data.get("program"),
                "activity": data.get("activity"),
            })

        fill_pdf_contract(PDF_intensive_TEMPLATE_PATH, outfile, data)
        generated_intensive_files.append(outfile)

len(generated_intensive_files), generated_intensive_files[:10]

[DEBUG] Drumm, Rene D., Summer 2026 – BeginDate=Timestamp('2026-05-04 00:00:00') -> 05/04/26, EndDate=Timestamp('2027-02-24 00:00:00') -> 02/24/27, course1_code='MSSN820'
[DEBUG] Duffis Gordon, Daniel M., Summer 2026 – BeginDate=Timestamp('2026-05-11 00:00:00') -> 05/11/26, EndDate=Timestamp('2027-02-24 00:00:00') -> 02/24/27, course1_code='MSSN829-001'
[DEBUG] Pavel, Zubkov, Summer 2026 – BeginDate=Timestamp('2026-05-11 00:00:00') -> 05/11/26, EndDate=Timestamp('2027-02-27 00:00:00') -> 02/27/27, course1_code='MSSN838-001'
[DEBUG] Sanou, Boubakar, Summer 2026 – BeginDate=Timestamp('2026-05-11 00:00:00') -> 05/11/26, EndDate=Timestamp('2027-02-24 00:00:00') -> 02/24/27, course1_code='MSSN885-001'


(4,
 [PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-02_Summer_intensive_Rene_D._Drumm_MSSN820_row6.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-02_Summer_intensive_Daniel_M._Duffis_Gordon_MSSN829-001_row5.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-02_Summer_intensive_Zubkov_Pavel_MSSN838-001_row3.pdf'),
  PosixPath('Step3_output_contract-production_PDF-CONTRACTS_filled/2026-02_Summer_intensive_Boubakar_Sanou_MSSN885-001_row4.pdf')])

# Do you want to proceed with **DocuSign**?

In [23]:
# ============================================================
# MANUAL CHECKPOINT — confirm you inspected the generated PDFs
# ============================================================

print("\n" + "="*70)
print("IMPORTANT: PLEASE REVIEW YOUR CONTRACTS BEFORE SENDING TO DOCUSIGN")
print("="*70)
print("Please inspect all generated contracts manually.")
print("Remove any contract you do NOT want to send to DocuSign.")
print("Only continue once your inspection is fully complete.\n")

resp1 = input("Type YES in CAPITAL LETTERS to confirm you have completed your inspection: ").strip()

if resp1 != "YES":
    raise SystemExit(
        "Process aborted. Contracts were NOT sent to DocuSign.\n"
        "Inspection confirmation was not provided."
    )

print("\nSecond confirmation required before sending contracts.\n")

resp2 = input(
    "Type YES again in CAPITAL LETTERS to confirm that ALL remaining contracts should be sent to DocuSign: "
).strip()

if resp2 != "YES":
    raise SystemExit(
        "Process aborted. Contracts were NOT sent to DocuSign.\n"
        "Final confirmation was not provided."
    )

print("\nWe are now sending your contracts to DocuSign...\n")


IMPORTANT: PLEASE REVIEW YOUR CONTRACTS BEFORE SENDING TO DOCUSIGN
Please inspect all generated contracts manually.
Remove any contract you do NOT want to send to DocuSign.
Only continue once your inspection is fully complete.



SystemExit: Process aborted. Contracts were NOT sent to DocuSign.
Inspection confirmation was not provided.

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# DocuSign Procedure

## DocuSign Authentification

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote
from docusign_esign import ApiClient

# ---------------------------
# SET THESE VALUES
# ---------------------------
os.environ["DS_CLIENT_ID"] = "49b37080-f006-4a44-b1d8-df32f8c63d67"  # Integration Key
os.environ["DS_USER_ID"] = "b37e3f09-e0f4-4992-8830-a52f994574a3"    # Impersonated User GUID
os.environ["DS_PRIVATE_KEY_PATH"] = "0_source_files/DocuSign/DocuSignPrivKey.pem"
os.environ["DS_AUTH_SERVER"] = "account-d.docusign.com"  # Demo (use account.docusign.com for production)
os.environ["DS_SCOPES"] = "signature impersonation"

# Pick ONE redirect URI that you have registered in Apps & Keys:
# (Either of these works since you added both.)
os.environ["DS_REDIRECT_URI"] = "https://developers.docusign.com/platform/auth/consent"
# os.environ["DS_REDIRECT_URI"] = "https://www.docusign.com"


def build_docusign_consent_url():
    """ f
    Prints a one-time consent URL. Open it in your browser, log in, click Allow.
    After that, JWT will stop returning {"error":"consent_required"} (for this env + user + integration key).
    """
    client_id = os.environ["DS_CLIENT_ID"]
    auth_server = os.environ["DS_AUTH_SERVER"]
    redirect_uri = os.environ["DS_REDIRECT_URI"]

    consent_url = (
        f"https://{auth_server}/oauth/auth"
        f"?response_type=code"
        f"&scope={quote(os.environ['DS_SCOPES'])}"
        f"&client_id={quote(client_id)}"
        f"&redirect_uri={quote(redirect_uri, safe='')}"
    )
    return consent_url


def docusign_jwt_login():
    client_id = os.environ["DS_CLIENT_ID"]
    user_id = os.environ["DS_USER_ID"]
    auth_server = os.environ["DS_AUTH_SERVER"]
    scopes = os.environ["DS_SCOPES"].split()
    private_key_path = Path(os.environ["DS_PRIVATE_KEY_PATH"])

    if not private_key_path.exists():
        raise FileNotFoundError(f"Private key not found at: {private_key_path}")

    api_client = ApiClient()
    api_client.host = f"https://{auth_server}"

    try:
        token = api_client.request_jwt_user_token(
            client_id=client_id,
            user_id=user_id,
            oauth_host_name=auth_server,
            private_key_bytes=private_key_path.read_bytes(),
            expires_in=3600,
            scopes=scopes
        )
    except Exception as e:
        # If consent is missing, print the consent URL immediately
        msg = str(e)
        if "consent_required" in msg:
            print("\nDocuSign returned: consent_required")
            print("Open this URL in your browser, log in, click Allow, then rerun this cell:\n")
            print(build_docusign_consent_url())
            raise
        raise

    access_token = token.access_token
    api_client.set_default_header("Authorization", f"Bearer {access_token}")

    user_info = api_client.get_user_info(access_token)
    acct = next((a for a in user_info.accounts if getattr(a, "is_default", False)), user_info.accounts[0])

    api_client.host = f"{acct.base_uri}/restapi"

    print("Authenticated successfully.")
    print("Account ID:", acct.account_id)
    print("Base URI:", acct.base_uri)

    return api_client, acct.account_id


# --- Run login (if consent is missing, you'll get a printed consent URL) ---
api_client, account_id = docusign_jwt_login()

## Template Method

In [ ]:
# Block 1 — Batch processor (folder-first) — UPDATED for CourseNum + CourseSection in filename
# + includes choose_template_id_for_pdf + template_roles_from_df_row + send_envelope_with_template_and_pdf
# + UPDATED: HRemail is now a REAL final signer (Signature5, routing_order=5)
#          (no more "fill Signature5 text tab" logic)

from pathlib import Path
import pandas as pd
import re
import base64

from docusign_esign import (
    EnvelopesApi,
    EnvelopeDefinition,
    TemplateRole,
    Document,
)

PDF_DIR = Path("Step3_output_contract-production_PDF-CONTRACTS_filled")

# ---------------------------------------------------------
# Parse NEW filename pattern (UPDATED):
#   YYYY-SS_Term_type_First_Last_TOKENS_rowNN[_partN].pdf
# where TOKENS looks like: OTST500-01+NTST540-02 (or single token)
# ---------------------------------------------------------
def parse_contract_pdf_stem(stem: str):
    """
    Supports:
      2026-02_Summer_intensive_Oliver_Glanz_OTST500-01+NTST540-02_row14
      2026-02_Summer_standard_Oliver_Glanz_OTST500-001_row12
      2026-03_Fall_standard_Angelika_Kaiser_CHIS612_row5
      ..._row14_part2  (optional suffix)

    Returns dict with: year(int), sem(str), term(str), ptype(str),
                       who(str), tokens(str), row_tag(str|None), part(int|None)
    """
    m = re.match(
        r"^(?P<year>\d{4})-(?P<sem>\d{2})_(?P<term>Spring|Summer|Fall)_(?P<ptype>standard|intensive)_(?P<rest>.+)$",
        stem,
        re.IGNORECASE
    )
    if not m:
        return None

    d = m.groupdict()
    year = int(d["year"])
    sem = d["sem"]
    term = d["term"].capitalize()
    ptype = d["ptype"].lower()
    rest = d["rest"]

    # Split rest into underscore parts for robust handling of _rowNN and _partN
    parts = rest.split("_")
    if len(parts) < 2:
        return None

    # optional trailing _partN
    part = None
    if parts and re.fullmatch(r"part\d+", parts[-1], re.IGNORECASE):
        part = int(re.sub(r"(?i)part", "", parts[-1]))
        parts = parts[:-1]

    # optional trailing _rowNN (this is the key fix)
    row_tag = None
    if parts and re.fullmatch(r"row\d+", parts[-1], re.IGNORECASE):
        row_tag = parts[-1]
        parts = parts[:-1]

    # After removing row/part: last part is tokens, rest is who
    if len(parts) < 2:
        return None

    tokens = parts[-1].strip()
    who = "_".join(parts[:-1]).strip()

    if not who or not tokens:
        return None

    return {
        "year": year,
        "sem": sem,
        "term": term,
        "ptype": ptype,
        "who": who,
        "tokens": tokens,
        "row_tag": row_tag,
        "part": part,
    }

# ---------------------------------------------------------
# Helper: choose template based on filename
# ---------------------------------------------------------
def choose_template_id_for_pdf(
    pdf_path: Path,
    template_id_standard: str,
    template_id_intensive: str
) -> str:
    info = parse_contract_pdf_stem(pdf_path.stem)
    if info and info.get("ptype") in ("standard", "intensive"):
        return template_id_standard if info["ptype"] == "standard" else template_id_intensive

    # Fallback if filename doesn't parse for some reason
    name = pdf_path.name.lower()
    if "intensive" in name:
        return template_id_intensive
    if "standard" in name:
        return template_id_standard

    raise ValueError(f"Cannot determine template type from filename: {pdf_path.name}")

# ---------------------------------------------------------
# Helper: build template roles from df row
# (Role names MUST match your DocuSign template roles)
# UPDATED: adds HR as Signature5 with routing_order=5 (FINAL SIGNER)
# ---------------------------------------------------------
def template_roles_from_df_row(row) -> list[dict]:
    def _clean_email(v):
        if v is None:
            return ""
        s = str(v).strip()
        return "" if s.lower() == "nan" else s

    instructor_email = _clean_email(row.get("Email", ""))
    chair_email      = _clean_email(row.get("ChairEmail", ""))
    dean_email       = _clean_email(row.get("DeanEmail", ""))
    vp_email         = _clean_email(row.get("VPFinanceEmail", ""))
    hr_email         = _clean_email(row.get("HRemail", ""))   # <-- NEW (final signer)

    missing = [k for k, v in {
        "Email": instructor_email,
        "ChairEmail": chair_email,
        "DeanEmail": dean_email,
        "VPFinanceEmail": vp_email,
        "HRemail": hr_email,
    }.items() if not v]

    if missing:
        who = row.get("Instructor", "<unknown>")
        yr  = row.get("Year", "")
        sem = row.get("Semester", "")
        raise ValueError(
            f"Missing required signer email(s): {missing} "
            f"for Instructor='{who}' ({sem} {yr}). "
            f"Fix your df_courses mapping for these columns."
        )

    return [
        {"role": "Signature1", "email": instructor_email, "name": instructor_email, "routing_order": "1"},
        {"role": "Signature2", "email": chair_email,      "name": chair_email,      "routing_order": "2"},
        {"role": "Signature3", "email": dean_email,       "name": dean_email,       "routing_order": "3"},
        {"role": "Signature4", "email": vp_email,         "name": vp_email,         "routing_order": "4"},
        {"role": "Signature5", "email": hr_email,         "name": hr_email,         "routing_order": "5"},  # HR LAST
    ]

# ---------------------------------------------------------
# Helper: create envelope from template + attach PDF
# UPDATED: no HR text-tab logic; HR is now a real role in template_roles
# ---------------------------------------------------------
def send_envelope_with_template_and_pdf(
    api_client,
    account_id: str,
    template_id: str,
    pdf_path: Path,
    roles: list[dict],
    email_subject: str,
    status: str = "created",  # "created" for drafts, "sent" to email out
) -> str:
    pdf_b64 = base64.b64encode(pdf_path.read_bytes()).decode("ascii")
    doc = Document(
        document_base64=pdf_b64,
        name=pdf_path.name,
        file_extension="pdf",
        document_id="1",
    )

    template_roles = []
    for r in roles:
        template_roles.append(
            TemplateRole(
                role_name=r["role"],
                name=r.get("name") or r["email"],
                email=r["email"],
                routing_order=str(r.get("routing_order", "1")),
            )
        )

    env = EnvelopeDefinition(
        template_id=template_id,
        template_roles=template_roles,
        documents=[doc],
        email_subject=email_subject,
        status=status,
    )

    envelopes_api = EnvelopesApi(api_client)
    result = envelopes_api.create_envelope(account_id, envelope_definition=env)
    return result.envelope_id

# ---------------------------------------------------------
# Normalizers
# ---------------------------------------------------------
def norm_pdf_who(who: str) -> str:
    s = str(who).lower().replace(".", "")
    s = re.sub(r"[_\s]+", "", s)
    return s

def norm_df_instr(instr: str) -> str:
    s = str(instr).lower().replace(",", " ").replace(".", "")
    parts = s.split()
    last = parts[0] if len(parts) > 0 else ""
    first = parts[1] if len(parts) > 1 else ""
    middle = parts[2] if len(parts) > 2 else ""
    return first + middle + last

def term_from_df_semester(val: str) -> str:
    v = str(val).lower()
    if "spring" in v: return "Spring"
    if "summer" in v: return "Summer"
    if "fall" in v:   return "Fall"
    return ""

def norm_course_token(t: str) -> str:
    return re.sub(r"\s+", "", str(t).strip().lower())

def tokens_from_df_row(row) -> str:
    dept = str(row.get("Dept", "")).strip()
    num  = str(row.get("CourseNum", "")).strip()
    sec  = str(row.get("CourseSection", "")).strip()

    base = f"{dept}{num}".strip()
    if sec and sec.lower() != "nan":
        sec = re.sub(r"\s+", "", sec)
        return f"{base}-{sec}"
    return base

def normalize_token_set(token_str: str) -> tuple:
    parts = [norm_course_token(x) for x in str(token_str).split("+") if str(x).strip()]
    return tuple(sorted(dict.fromkeys(parts)))

# ---------------------------------------------------------
# Prep df_courses for robust matching
# ---------------------------------------------------------
def prepare_df_courses_for_matching(df_courses: pd.DataFrame) -> pd.DataFrame:
    df = df_courses.copy()
    df["year_int"] = df["Year"].astype(str).str.extract(r"(\d{4})")[0].astype(float).astype("Int64")
    df["term"] = df["Semester"].apply(term_from_df_semester)
    df["name_key"] = df["Instructor"].apply(norm_df_instr)
    df["course_token"] = df.apply(tokens_from_df_row, axis=1).apply(norm_course_token)
    return df

DF_MATCH = prepare_df_courses_for_matching(df_courses)

# ---------------------------------------------------------
# Match PDF -> df rows using year+term+token set (+ instructor)
# ---------------------------------------------------------
def match_pdf_to_df_rows_for_tokens(info: dict, df_match: pd.DataFrame) -> pd.DataFrame:
    pdf_name_key = norm_pdf_who(info["who"])
    pdf_tokens = normalize_token_set(info["tokens"])
    if not pdf_tokens:
        raise ValueError(f"No tokens parsed from filename: {info}")

    cand = df_match[
        (df_match["year_int"] == info["year"]) &
        (df_match["term"] == info["term"]) &
        (df_match["name_key"] == pdf_name_key)
    ].copy()

    if cand.empty:
        return cand

    cand = cand[cand["course_token"].isin(set(pdf_tokens))].copy()
    return cand

def match_pdf_to_df_row(pdf_path: Path, df_match: pd.DataFrame) -> pd.Series:
    info = parse_contract_pdf_stem(pdf_path.stem)
    if info is None:
        raise ValueError(f"Filename does not match expected pattern: {pdf_path.name}")

    matches = match_pdf_to_df_rows_for_tokens(info, df_match)

    if matches.empty:
        raise ValueError(
            f"No match for PDF '{pdf_path.name}'. "
            f"(year={info['year']}, term={info['term']}, who={info['who']}, tokens={info['tokens']})"
        )

    # Safety: ensure signer emails are consistent across matched rows
    email_cols = ["Email", "ChairEmail", "DeanEmail", "VPFinanceEmail", "HRemail"]
    for c in email_cols:
        if c in matches.columns:
            uniq = matches[c].astype(str).str.strip().fillna("").unique().tolist()
            uniq = [u for u in uniq if u and u.lower() != "nan"]
            if len(set(uniq)) > 1:
                raise ValueError(
                    f"Ambiguous signer emails across matched rows for '{pdf_path.name}' in column '{c}': {set(uniq)}"
                )

    return matches.iloc[0]

# ---------------------------------------------------------
# Batch send
# ---------------------------------------------------------
def process_all_pdfs_in_folder(
    api_client,
    account_id: str,
    pdf_dir: Path,
    df_match: pd.DataFrame,
    template_id_standard: str,
    template_id_intensive: str,
    status: str = "created",  # "created" for drafts, "sent" to email out
) -> pd.DataFrame:

    pdfs = sorted(pdf_dir.glob("*.pdf"))
    if not pdfs:
        raise FileNotFoundError(f"No PDFs found in {pdf_dir}")

    log_rows = []

    for pdf_path in pdfs:
        entry = {
            "pdf_file": pdf_path.name,
            "pdf_path": str(pdf_path),
            "template_type": "",
            "template_id": "",
            "matched_instructor": "",
            "matched_year": "",
            "matched_term": "",
            "matched_tokens": "",
            "matched_rowtag": "",     # helps debug rowNN suffix
            "matched_hr_email": "",   # NEW: log HR signer email
            "envelope_id": "",
            "status": status,
            "error": "",
        }

        try:
            info = parse_contract_pdf_stem(pdf_path.stem)
            if info is None:
                raise ValueError(f"Filename does not match expected pattern: {pdf_path.name}")

            entry["matched_tokens"] = info["tokens"]
            entry["matched_rowtag"] = info.get("row_tag") or ""

            # choose template
            t_id = choose_template_id_for_pdf(pdf_path, template_id_standard, template_id_intensive)
            entry["template_id"] = t_id
            entry["template_type"] = info["ptype"]

            # match df row
            row = match_pdf_to_df_row(pdf_path, df_match)
            entry["matched_instructor"] = row.get("Instructor", "")
            entry["matched_year"] = row.get("Year", "")
            entry["matched_term"] = row.get("Semester", "")

            # build template roles (HR is Signature5 and will sign LAST)
            roles = template_roles_from_df_row(row)
            entry["matched_hr_email"] = str(row.get("HRemail", "")).strip()

            # send / create draft
            env_id = send_envelope_with_template_and_pdf(
                api_client=api_client,
                account_id=account_id,
                template_id=t_id,
                pdf_path=pdf_path,
                roles=roles,
                email_subject=f"Contract for signature: {pdf_path.stem}",
                status=status,
            )
            entry["envelope_id"] = env_id

        except Exception as e:
            entry["error"] = f"{type(e).__name__}: {e}"

        log_rows.append(entry)

    return pd.DataFrame(log_rows)

In [ ]:
from docusign_esign import TemplatesApi

TEMPLATE_NAME_STANDARD = "Standard Contract"
TEMPLATE_NAME_INTENSIVE = "Intensive Contract"

def get_template_id_by_name(api_client, account_id: str, template_name: str) -> str:
    templates_api = TemplatesApi(api_client)
    tpl_list = templates_api.list_templates(account_id)

    items = getattr(tpl_list, "envelope_templates", None) or []
    for t in items:
        if (t.name or "").strip().lower() == template_name.strip().lower():
            return t.template_id

    # If not found, print what *is* available (helps debug)
    available = [t.name for t in items]
    raise ValueError(f"Template not found: '{template_name}'. Available templates: {available}")

TEMPLATE_ID_STANDARD  = get_template_id_by_name(api_client, account_id, TEMPLATE_NAME_STANDARD)
TEMPLATE_ID_INTENSIVE = get_template_id_by_name(api_client, account_id, TEMPLATE_NAME_INTENSIVE)

print("Standard template_id :", TEMPLATE_ID_STANDARD)
print("Intensive template_id:", TEMPLATE_ID_INTENSIVE)

In [ ]:
# Block 2 — Run it (drafts first): This creates drafts for every PDF in the folder:

df_log = process_all_pdfs_in_folder(
    api_client=api_client,
    account_id=account_id,
    pdf_dir=PDF_DIR,
    df_match=DF_MATCH,
    template_id_standard=TEMPLATE_ID_STANDARD,
    template_id_intensive=TEMPLATE_ID_INTENSIVE,
    status="created"   # change to "sent" after you confirm drafts look right
    #status="sent"   # change to "sent" after you confirm drafts look right
)

df_log

In [ ]:
# Block 3 — Show only failures (if any)

df_log[df_log["error"] != ""]